In [52]:
import torch
import torchtt

In [53]:
layer = torchtt.nn.CompressedTTLayer([3,3,3,3], [2,2,2,2], [1,2,2,2,1], [1,4,4,4,1])

t_in = torchtt.random([3,3,3,3], [1,2,3,4,1], dtype=torch.float32)

%time layer.forward(t_in)

CPU times: user 3.17 ms, sys: 17 μs, total: 3.19 ms
Wall time: 3.2 ms


TT with sizes and ranks:
N = [2, 2, 2, 2]
R = [1, 2, 4, 4, 1]

Device: cpu, dtype: torch.float32
#entries 60 compression 3.75

In [54]:
# Demonstrate automatic differentiation through the CompressedTTLayer.
# The forward pass involves SVD-based truncation, but gradients still
# flow back to the layer's trainable cores and biases.

# Forward pass
output = layer(t_in)

# Scalar loss: sum of all entries in the output TT
loss = sum(c.sum() for c in output.cores)

# Backpropagate
loss.backward()

# Inspect gradients on the layer's weight cores
for k, core in enumerate(layer.cores):
    print(f'Core {k} | shape {list(core.shape)} | grad norm {core.grad.norm().item():.6f}')

# Gradient descent step
lr = 1e-3
with torch.no_grad():
    for p in layer.parameters():
        p -= lr * p.grad
        p.grad.zero_()

print('\nGradient step done.')

Core 0 | shape [1, 2, 3, 2] | grad norm 4.243818
Core 1 | shape [2, 2, 3, 2] | grad norm 6.735900
Core 2 | shape [2, 2, 3, 2] | grad norm 8.651754
Core 3 | shape [2, 2, 3, 1] | grad norm nan

Gradient step done.
